## Environment Setup (run once before starting)

### Option A — VS Code (recommended)
1. Open a terminal in the project root
2. Create a virtual environment:
   
3. Activate it:
   - **Windows:** 
   - **Mac/Linux:** 
4. Install all dependencies:
   
5. In VS Code:  → *Python: Select Interpreter* → choose 
6. Open this notebook and select the  kernel in the top-right

### Option B — install directly in this notebook
> Uncomment and run the cell below (only once).


In [ ]:
# Uncomment to install all course dependencies (run only once)
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])


# Session 1 — Walk-Forward Optimisation & MADL

**Course:** AI in Investment Strategies  
**Instructor:** Robert Ślepaczuk, WNE University of Warsaw

## Objectives
1. Implement a dual-MA momentum/contrarian strategy on SPY
2. Run a proper Walk-Forward Optimisation (WFO)
3. Evaluate with Sharpe, MD, Calmar, and **MADL / GMADL**
4. Transaction cost sensitivity analysis
5. Statistical significance testing (SR t-test, multiple comparisons)
6. Visualise equity curves, IS/OOS scatter, and MADL distributions

### Key references
- Michańków, Sakowski & Ślepaczuk (2024) — MADL, *Journal of Computational Science* 81
- Mroziewicz & Ślepaczuk (2026) — Double OOS, *arXiv:2602.10785*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
from itertools import product
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'momentum': '#003366', 'contrarian': '#c41230', 'bh': '#888888'}

## 1. Data

In [ ]:
ticker = 'SPY'
df = yf.download(ticker, start='2005-01-01', end='2024-12-31', auto_adjust=True)
prices = df['Close'].squeeze()
returns = prices.pct_change().dropna()
print(f'Downloaded {len(prices)} trading days of {ticker} ({prices.index[0].date()} – {prices.index[-1].date()})')

## 2. Performance Metrics

In [ ]:
def sharpe(r, ann=252):
    if r.std() == 0: return 0.0
    return r.mean() / r.std() * np.sqrt(ann)

def asd(r, ann=252):
    """Annualised Standard Deviation."""
    return r.std() * np.sqrt(ann)

def max_drawdown(r):
    w = (1 + r).cumprod()
    return ((w - w.cummax()) / w.cummax()).min()

def mld(r):
    """Maximum Loss Duration (in years): longest peak-to-recovery period."""
    w = (1 + r).cumprod()
    peak = w.cummax()
    in_drawdown = w < peak
    max_dur = 0
    cur_dur = 0
    for val in in_drawdown:
        if val:
            cur_dur += 1
            max_dur = max(max_dur, cur_dur)
        else:
            cur_dur = 0
    return max_dur / 252  # convert to years

def cagr(r, ann=252):
    total = (1 + r).prod()
    years = len(r) / ann
    return total ** (1 / years) - 1

def calmar(r, ann=252):
    mdd_val = abs(max_drawdown(r))
    return cagr(r, ann) / mdd_val if mdd_val > 0 else np.nan

def ir_star(r):
    """IR* = ARC / ASD  (return-to-volatility ratio)."""
    vol = asd(r)
    return cagr(r) / vol if vol > 0 else np.nan

def ir_double_star(r):
    """IR** = ARC^2 * sign(ARC) / (ASD * MD)."""
    c = cagr(r)
    vol = asd(r)
    mdd_val = abs(max_drawdown(r))
    if vol == 0 or mdd_val == 0: return np.nan
    return (c**2 * np.sign(c)) / (vol * mdd_val)

def ir_triple_star(r):
    """IR*** = ARC^3 / (ASD * MD * MLD), MLD in years."""
    c = cagr(r)
    vol = asd(r)
    mdd_val = abs(max_drawdown(r))
    mld_val = mld(r)
    if vol == 0 or mdd_val == 0 or mld_val == 0: return np.nan
    return c**3 / (vol * mdd_val * mld_val)

def madl(r_hat, r_actual):
    """
    Mean Absolute Directional Loss (JoCS 2024, eq. 6). Minimize.
    MADL < 0: correct directions, magnitude-weighted (profitable).
    MADL ~ 0: random (no directional skill).
    MADL > 0: wrong directions dominate (losing money).
    Michankow, Sakowski & Slepaczuk (J. Comp. Sci. 2024).
    doi:10.1016/j.jocs.2024.102375
    """
    rh = np.asarray(r_hat, dtype=float)
    ra = np.asarray(r_actual, dtype=float)
    return float((-np.sign(rh * ra) * np.abs(ra)).mean())

def gmadl(r_hat, r_actual, a=4.0, b=1.0):
    """
    Generalised MADL (arXiv:2412.18405v1, eq. 2). Minimize.
    Smooth differentiable version: sigmoid replaces sign().
    a: slope parameter (sharpness of sign approximation).
    b: magnitude power (return weighting).
    Michankow, Sakowski & Slepaczuk (arXiv:2412.18405v1, 2024).
    doi:10.48550/arXiv.2412.18405
    """
    rh = np.asarray(r_hat, dtype=float)
    ra = np.asarray(r_actual, dtype=float)
    sigmoid = 1.0 / (1.0 + np.exp(-a * rh * ra))
    return float((-1.0 * (sigmoid - 0.5) * np.abs(ra) ** b).mean())

def hit_rate(r_hat, r_actual):
    return float((np.sign(r_hat) == np.sign(r_actual)).mean())

def sharpe_ttest(r):
    """H0: SR <= 0 | returns (t-stat, p-value one-tailed)."""
    sr_daily = r.mean() / r.std()
    se = np.sqrt((1 + 0.5 * sr_daily**2) / len(r))
    t = sr_daily / se
    p = stats.t.sf(t, df=len(r)-1)
    return t, p

def full_metrics(r, label=""):
    t, p = sharpe_ttest(r)
    c = cagr(r)
    return {
        "ARC":   f"{c:.1%}",
        "SR":     f"{(c - 0) / asd(r):.2f}",
        "IR*":    f"{ir_star(r):.2f}",
        "IR**":   f"{ir_double_star(r):.2f}" if not np.isnan(ir_double_star(r)) else "n/a",
        "IR***":  f"{ir_triple_star(r):.2f}" if not np.isnan(ir_triple_star(r)) else "n/a",
        "MD":    f"{max_drawdown(r):.1%}",
        "MLD(y)": f"{mld(r):.2f}",
        "Calmar": f"{calmar(r):.2f}",
        "SR p":   f"{p:.3f}",
    }

# Sanity checks — with correct formula: perfect < 0, all-wrong > 0
dummy = pd.Series([0.01, -0.02, 0.03, -0.04, 0.01])
perfect, all_wrong = dummy.copy(), -dummy.copy()
print(f"MADL  perfect={madl(perfect, dummy):.4f} (expect < 0) | all-wrong={madl(all_wrong, dummy):.4f} (expect > 0)")
print(f"GMADL perfect={gmadl(perfect, dummy):.4f} (expect < 0) | all-wrong={gmadl(all_wrong, dummy):.4f} (expect > 0)")

## 3. Dual-MA Signal & Strategy Returns

In [ ]:
def dual_ma_signal(prices, n1, n2, direction='momentum', shift=1):
    fast = prices.rolling(n1).mean()
    slow = prices.rolling(n2).mean()
    raw  = np.sign(fast - slow).shift(shift)  # shift=1: no look-ahead (default)
    return -raw if direction == 'contrarian' else raw

def strategy_returns(prices, n1, n2, direction='momentum', tc=0.001, shift=1):
    r   = prices.pct_change()
    sig = dual_ma_signal(prices, n1, n2, direction, shift=shift)
    gross    = sig * r
    turnover = sig.diff().abs() / 2
    net      = gross - turnover * tc
    return net.dropna()

## 4. Walk-Forward Optimisation

In [ ]:
def run_wfo(
    prices,
    is_days=252,
    oos_days=63,
    n1_range=range(2, 51, 5),
    n2_range=range(50, 253, 25),
    direction='momentum',
    tc=0.001,
    criterion='sharpe',
):
    """
    Walk-Forward Optimisation.
    criterion: 'sharpe' | 'madl' (optimise negative MADL)
    Returns: (oos_returns, diagnostics_df)
    """
    oos_returns, records = [], []
    global_ret = prices.pct_change().dropna()

    for start in range(is_days, len(prices) - oos_days, oos_days):
        is_prices  = prices.iloc[start - is_days : start]
        oos_prices = prices.iloc[start : start + oos_days]

        best_score, best_p = -np.inf, None
        for n1, n2 in product(n1_range, n2_range):
            if n1 >= n2:
                continue
            r_is = strategy_returns(is_prices, n1, n2, direction, tc)
            if len(r_is) < 30:
                continue
            score = sharpe(r_is) if criterion == 'sharpe' else -madl(r_is, r_is)
            if score > best_score:
                best_score, best_p = score, (n1, n2)

        if best_p is None:
            continue

        n1, n2 = best_p
        # use tail of IS for warm-up so MA is properly initialised
        context = pd.concat([is_prices.iloc[-n2:], oos_prices])
        oos_r   = strategy_returns(context, n1, n2, direction, tc).iloc[-len(oos_prices):]

        # align with actual returns for MADL computation
        actual = global_ret.reindex(oos_r.index)

        oos_returns.append(oos_r)
        records.append({
            'oos_start':   oos_prices.index[0],
            'n1': n1, 'n2': n2,
            'is_sharpe':   best_score if criterion == 'sharpe' else np.nan,
            'oos_sharpe':  sharpe(oos_r),
            'oos_madl':    madl(oos_r, actual),
            'oos_gmadl':   gmadl(oos_r, actual),
            'oos_hit':     hit_rate(oos_r, actual),
            'oos_cagr':    cagr(oos_r),
        })

    curve = pd.concat(oos_returns).sort_index() if oos_returns else pd.Series(dtype=float)
    return curve, pd.DataFrame(records)

## 5. Run Strategies

In [ ]:
print('Running WFO momentum ...')
mom_oos, mom_diag = run_wfo(prices, direction='momentum')

print('Running WFO contrarian ...')
con_oos, con_diag = run_wfo(prices, direction='contrarian')

bh = returns.reindex(mom_oos.index)
print(f'\nOOS period: {mom_oos.index[0].date()} – {mom_oos.index[-1].date()} ({len(mom_oos)} days)')

## 6. Performance Summary Table

In [ ]:
summary = {name: full_metrics(r)
           for name, r in [('Buy & Hold', bh),
                            ('Momentum WFO', mom_oos),
                            ('Contrarian WFO', con_oos)]}
pd.DataFrame(summary).T

## 7. Equity Curves & IS/OOS Diagnostic

In [ ]:
import os; os.makedirs('../data', exist_ok=True)
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)

# ── Equity curves ──
ax0 = fig.add_subplot(gs[0, :])
for name, r, c in [('Buy & Hold', bh, COLORS['bh']),
                   ('Momentum WFO', mom_oos, COLORS['momentum']),
                   ('Contrarian WFO', con_oos, COLORS['contrarian'])]:
    (1 + r).cumprod().plot(ax=ax0, label=name, color=c, linewidth=1.6)
ax0.set_title('OOS Equity Curves — SPY (TC = 10 bps)', fontsize=12)
ax0.set_ylabel('Cumulative Return')
ax0.legend()

# ── IS vs OOS Sharpe ──
ax1 = fig.add_subplot(gs[1, 0])
ax1.scatter(mom_diag['is_sharpe'], mom_diag['oos_sharpe'],
            c=COLORS['momentum'], alpha=0.75, s=45, label='Momentum')
ax1.scatter(con_diag['is_sharpe'], con_diag['oos_sharpe'],
            c=COLORS['contrarian'], alpha=0.75, s=45, marker='^', label='Contrarian')
lim = [-3, 5]
ax1.plot(lim, lim, 'k--', lw=0.8, label='IS = OOS')
ax1.plot(lim, [0.5*x for x in lim], color='gray', ls=':', lw=0.8, label='0.5 ratio')
ax1.axhline(0, color='red', lw=0.5, ls='--')
ax1.set_xlabel('IS Sharpe'); ax1.set_ylabel('OOS Sharpe')
ax1.set_title('IS vs OOS Sharpe (overfitting diagnostic)')
ax1.legend(fontsize=8)

# ── MADL distribution ──
ax2 = fig.add_subplot(gs[1, 1])
mom_diag['oos_madl'].hist(ax=ax2, bins=12, alpha=0.7, color=COLORS['momentum'],
                           edgecolor='white', label='Momentum')
con_diag['oos_madl'].hist(ax=ax2, bins=12, alpha=0.55, color=COLORS['contrarian'],
                           edgecolor='white', label='Contrarian')
ax2.axvline(0, color='black', lw=1.2, ls='--', label='Random (MADL≈0)')
ax2.set_xlabel('MADL per OOS window')
ax2.set_title('MADL Distribution (negative = profitable, positive = losing)')
ax2.legend(fontsize=8)

plt.savefig('../data/session1_full_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 8. Transaction Cost Sensitivity Analysis

Key question: at what cost level does the strategy stop beating Buy & Hold?

In [ ]:
tc_levels = [0, 0.0002, 0.0005, 0.001, 0.002, 0.003, 0.005, 0.01]
tc_results = []
bh_sr = sharpe(bh)

for tc in tc_levels:
    m_oos, _ = run_wfo(prices, direction='momentum', tc=tc)
    c_oos, _ = run_wfo(prices, direction='contrarian', tc=tc)
    tc_results.append({
        'TC (bps)': round(tc * 10_000),
        'Mom SR':   round(sharpe(m_oos), 2),
        'Con SR':   round(sharpe(c_oos), 2),
        'B&H SR':   round(bh_sr, 2),
    })

tc_df = pd.DataFrame(tc_results).set_index('TC (bps)')
display(tc_df)

fig, ax = plt.subplots(figsize=(10, 4))
tc_df[['Mom SR', 'Con SR', 'B&H SR']].plot(
    ax=ax,
    color=[COLORS['momentum'], COLORS['contrarian'], COLORS['bh']],
    marker='o', linewidth=2)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Transaction Cost (bps one-way)')
ax.set_ylabel('Sharpe Ratio (OOS)')
ax.set_title('Strategy Sharpe vs. Transaction Cost Level')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Statistical Significance & Multiple Testing

Bonferroni correction for testing multiple strategies.

In [ ]:
alpha = 0.05

strategies_to_test = [
    ('Momentum WFO', mom_oos),
    ('Contrarian WFO', con_oos),
    ('Buy & Hold', bh),
]

K = len(strategies_to_test)
alpha_bonf = alpha / K  # Bonferroni correction

print(f'Testing {K} strategies | alpha={alpha} | Bonferroni alpha={alpha_bonf:.4f}\n')
print(f'{"Strategy":<22} {"SR":>6} {"t-stat":>8} {"p-value":>9} {"Sig (raw)":>12} {"Sig (Bonf)":>12}')
print('-' * 75)

for name, r in strategies_to_test:
    sr = sharpe(r)
    t, p = sharpe_ttest(r)
    sig_raw  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < alpha else 'n.s.'))
    sig_bonf = '***' if p < alpha_bonf/10 else ('*' if p < alpha_bonf else 'n.s.')
    print(f'{name:<22} {sr:>6.2f} {t:>8.2f} {p:>9.4f} {sig_raw:>12} {sig_bonf:>12}')

print(f'\nNote: * p<{alpha_bonf:.4f} after Bonferroni; *** p<{alpha_bonf/10:.5f}')

## 10. Double Out-of-Sample Validation

Implementation of the three-phase protocol from Mroziewicz & Ślepaczuk (arXiv:2602.10785, 2026).

In [ ]:
n = len(prices)
# Allocate: 50% IS | 25% OOS1 | 25% OOS2
is_end   = int(n * 0.50)
oos1_end = int(n * 0.75)

prices_is   = prices.iloc[:is_end]
prices_oos1 = prices.iloc[is_end:oos1_end]
prices_oos2 = prices.iloc[oos1_end:]

print(f'IS   : {prices_is.index[0].date()} – {prices_is.index[-1].date()}  ({len(prices_is)} days)')
print(f'OOS1 : {prices_oos1.index[0].date()} – {prices_oos1.index[-1].date()}  ({len(prices_oos1)} days)')
print(f'OOS2 : {prices_oos2.index[0].date()} – {prices_oos2.index[-1].date()}  ({len(prices_oos2)} days)')

# Phase 1: optimise on IS
best_is_sr, best_n1, best_n2 = -np.inf, None, None
for n1, n2 in product(range(2, 51, 5), range(50, 253, 25)):
    if n1 >= n2:
        continue
    r = strategy_returns(prices_is, n1, n2, 'momentum')
    if len(r) < 50:
        continue
    sr = sharpe(r)
    if sr > best_is_sr:
        best_is_sr, best_n1, best_n2 = sr, n1, n2

print(f'\nBest IS params: n1={best_n1}, n2={best_n2}, IS Sharpe={best_is_sr:.2f}')

# Phase 2: evaluate on OOS1 (parameter selection, no re-optimisation)
context1 = pd.concat([prices_is.iloc[-best_n2:], prices_oos1])
r_oos1   = strategy_returns(context1, best_n1, best_n2, 'momentum').iloc[-len(prices_oos1):]
print(f'OOS1 Sharpe:    {sharpe(r_oos1):.2f}  (IS/OOS1 ratio: {sharpe(r_oos1)/best_is_sr:.2f})')

# Phase 3: final validation on OOS2 — TOUCH EXACTLY ONCE
context2 = pd.concat([prices_oos1.iloc[-best_n2:], prices_oos2])
r_oos2   = strategy_returns(context2, best_n1, best_n2, 'momentum').iloc[-len(prices_oos2):]
bh_oos2  = returns.reindex(r_oos2.index)
print(f'OOS2 Sharpe:    {sharpe(r_oos2):.2f}  (final unbiased estimate)')
print(f'OOS2 B&H SR:    {sharpe(bh_oos2):.2f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (label, r_strat, r_bh, c) in zip(axes, [
    ('IS',   strategy_returns(prices_is, best_n1, best_n2, 'momentum'),
             returns.reindex(prices_is.index), COLORS['momentum']),
    ('OOS1', r_oos1,   returns.reindex(r_oos1.index),  COLORS['momentum']),
    ('OOS2', r_oos2,   bh_oos2,                         COLORS['contrarian']),
]):
    (1 + r_strat).cumprod().plot(ax=ax, color=c, label='Momentum', linewidth=1.5)
    (1 + r_bh).cumprod().plot(ax=ax, color='gray', label='B&H', linewidth=1.2, ls='--')
    ax.set_title(f'{label} Period')
    ax.legend(fontsize=8)

plt.suptitle('Double OOS Validation — Best Momentum Parameters', y=1.02)
plt.tight_layout()
plt.show()

## 11. Look-Ahead Bias — Live Demonstration

The line `np.sign(...).shift(1)` is what keeps a backtest honest: a signal computed from data
up to day *t* may only be **acted on from day *t+1***. With `shift=0` the position on day *t*
already "knows" day *t*'s close and then collects day *t*'s return — information that did not
exist yet.

We run the **same bug** (`shift=1` honest vs `shift=0` look-ahead) on **two different signals**
to expose a subtlety most courses miss:

1. **Slow dual-MA crossover** — the strategy this notebook actually optimises.
2. **Sign-of-return** — a naive signal whose value *is* the day's direction.

> **Key insight — severity depends on how directly the signal touches the leaked bar.**
> A 100-day moving average barely moves when one new bar is added, so removing `.shift(1)`
> changes the dual-MA results only **mildly** (and the effect shrinks as the windows get
> slower). But a *sign-of-return* signal literally contains the answer, so the same one-line
> bug turns it into a near-straight line up with an absurd Sharpe.
>
> Lesson: **never assume look-ahead is "small".** One missing `.shift(1)` can hide either a
> rounding error or an entire fake fortune — you cannot tell without checking.

In [ ]:
def signal_returns(sig, r, tc=0.001):
    """Net returns for an arbitrary position signal (same cost model as strategy_returns)."""
    sig = sig.reindex(r.index)
    return (sig * r - sig.diff().abs() / 2 * tc).dropna()

# ── Demo 1: slow dual-MA crossover (the strategy this notebook optimises) ──
n1_demo = best_n1 if ('best_n1' in globals() and best_n1) else 20
n2_demo = best_n2 if ('best_n2' in globals() and best_n2) else 100
ma_honest = strategy_returns(prices, n1_demo, n2_demo, 'momentum', shift=1)
ma_look   = strategy_returns(prices, n1_demo, n2_demo, 'momentum', shift=0)

# ── Demo 2: sign-of-return signal (value IS the day's direction) ──
sr_honest = signal_returns(np.sign(returns).shift(1), returns)  # trade on yesterday's sign
sr_look   = signal_returns(np.sign(returns),          returns)  # peek at today's sign

comparison = pd.DataFrame({
    f'dual-MA honest (n1={n1_demo},n2={n2_demo})': full_metrics(ma_honest),
    'dual-MA LOOK-AHEAD':                          full_metrics(ma_look),
    'sign-return honest':                          full_metrics(sr_honest),
    'sign-return LOOK-AHEAD':                       full_metrics(sr_look),
}).T
print('Same .shift bug on two signals — honest (shift=1) vs look-ahead (shift=0):\n')
display(comparison)

# Use ARC/ASD (= the table's "SR" = IR*) so narrative, legends and table all agree.
print(f'\nDual-MA   : SR {ir_star(ma_honest):.2f} -> {ir_star(ma_look):.2f}'
      f'  — mild, a slow average barely uses today\'s bar')
print(f'Sign-ret  : SR {ir_star(sr_honest):.2f} -> {ir_star(sr_look):.2f}'
      f'  — CATASTROPHIC: the signal literally contains the answer')
print('Both LOOK-AHEAD columns are FAKE. Severity depends on how directly the signal leaks.')

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))
(1 + ma_honest).cumprod().plot(ax=axL, color=COLORS['momentum'], lw=1.8,
                               label=f'honest  SR={ir_star(ma_honest):.2f}')
(1 + ma_look).cumprod().plot(ax=axL, color=COLORS['contrarian'], lw=1.8,
                             label=f'look-ahead  SR={ir_star(ma_look):.2f}')
axL.set_title(f'Slow dual-MA (n1={n1_demo}, n2={n2_demo}) — leak is mild')
axL.set_ylabel('Cumulative Return'); axL.legend()

(1 + sr_honest).cumprod().plot(ax=axR, color=COLORS['momentum'], lw=1.8,
                               label=f'honest  SR={ir_star(sr_honest):.2f}')
(1 + sr_look).cumprod().plot(ax=axR, color=COLORS['contrarian'], lw=1.8,
                             label=f'look-ahead  SR={ir_star(sr_look):.2f}')
axR.set_yscale('log')
axR.set_title('Sign-of-return — leak is catastrophic (log scale)')
axR.set_ylabel('Cumulative Return (log)'); axR.legend()

plt.suptitle('Look-Ahead Bias: same `.shift` bug, very different damage', y=1.02)
plt.tight_layout()
plt.show()